In [83]:
import os
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
from db_manager import DBManager
import pandas as pd
import numpy as np

df_team = pd.read_csv('../crawling_data/data_csv/team.csv', encoding='utf-8')
df_batter_offence = pd.read_csv('../crawling_data/data_csv/batter_offence.csv', encoding='utf-8')
df_batter_offence2 = pd.read_csv('../crawling_data/data_csv/batter_offence2.csv', encoding='utf-8')
df_batter_defence = pd.read_csv('../crawling_data/data_csv/batter_defence.csv',encoding='utf-8')
df_pitcher = pd.read_csv('../crawling_data/data_csv/pitcher.csv', encoding='utf-8')

In [74]:
def convert_ip(value):
    try:
        parts = str(value).split()
        if len(parts) == 1:
            return float(parts[0])
        v_num, v_str = parts
        v1, v2 = v_str.split('/')
        return float(v_num) + int(v1) / int(v2)
    except:
        return 0 

In [75]:
df = pd.DataFrame(df_team)
df_copy = df.copy()
df_team = df_copy.drop(columns = ['game' ,'winning_rate', 'games_behind', 'recent10'], axis=1)
df_team[['home_win', 'home_draw', 'home_lose']] = df_team['home'].str.split('-', expand=True).astype(int)
# expand=True : 분리된 값을 각각 컬럼으로 확장
df_team[['away_win', 'away_draw', 'away_lose']] = df_team['away'].str.split('-', expand=True).astype(int)
df_team = df_team.drop(columns=['home', 'away'], axis=1)
df_team.head()

,year,ranking,team,win,lose,draw,inarow,home_win,home_draw,home_lose,away_win,away_draw,away_lose
0,2022,1,SSG,88,52,4,4패,49,0,23,39,4,29
1,2022,2,키움,80,62,2,1승,39,1,32,41,1,30
2,2022,3,LG,87,55,2,1승,38,1,33,49,1,22
3,2022,4,KT,80,62,2,1패,40,1,31,40,1,31
4,2022,5,KIA,70,73,1,1패,31,0,41,39,1,32


In [76]:
df = pd.DataFrame(df_batter_offence)
df_copy=df.copy()
df_offence = df_copy.drop(['tb', 'sac'], axis = 1)
df_offence = df_offence.rename(columns= {'pa' : 'plate_appearance' , 'ab' : 'abat', 'r': 'scored'})
df_offence = df_offence.drop(columns=['Unnamed: 0'])
df_offence = df_offence.replace('-', np.nan)
df_offence[ df_offence.isnull().any(axis=1) ]
df_offence = df_offence.dropna()
df_offence.head()

,year,position,name,team,avg,game_cnt,plate_appearance,abat,scored,hit,2b,3b,home_run,rbi,sf
0,2022,포수,이병헌,삼성,0.750,3,4,4,1,3,1,0,0,1,0
1,2022,포수,권다결,KIA,0.500,2,2,2,0,1,0,0,0,1,0
2,2022,포수,김재성,삼성,0.335,63,185,161,16,54,10,0,3,26,3
3,2022,포수,안승한,두산,0.333,30,39,36,5,12,3,0,0,8,0
4,2022,포수,김선우,KIA,0.333,3,4,3,1,1,0,0,0,0,0


In [77]:
df = pd.DataFrame(df_batter_defence)
df_fielder = df[df['position'] != '포수']
df_fielder = df_fielder.drop(columns=['Unnamed: 0', 'pko', 'pb', 'sb', 'cs', 'cs_pct'])
df_fielder = df_fielder.rename(columns ={'inning_defence' : 'inning'})
df_fielder[df_fielder.isin(['-']).any(axis=1)]
df_fielder = df_fielder.replace('-', 0)
df_fielder['inning'] = df_fielder['inning'].apply(convert_ip).round(3)
df_fielder.head()

,year,position,name,team,detail_position,game,gama_starter,inning,error,po,assist,double_play,fpct
46,2022,내야수,양석환,두산,1루수,85,83,685.000,10,629,55,54,0.986
47,2022,내야수,김민성,LG,3루수,79,25,313.333,3,19,46,3,0.956
48,2022,내야수,김주원,NC,유격수,79,70,636.333,11,139,206,48,0.969
49,2022,내야수,정훈,롯데,1루수,77,74,614.667,2,561,43,49,0.997
50,2022,내야수,오영수,NC,1루수,76,63,553.667,4,486,40,50,0.992


In [78]:
df = pd.DataFrame(df_batter_defence)
df_catcher = df[df['position'] == '포수']
df_catcher = df_catcher.drop(columns=['Unnamed: 0', 'pko'])
df_catcher = df_catcher.rename(columns ={'inning_defence' : 'inning'})
df_catcher[ df_catcher.isin(['-']).any(axis=1) ]
df_catcher = df_catcher.replace('-', np.nan)
df_catcher = df_catcher.dropna()
df_catcher['inning'] = df_catcher['inning'].apply(convert_ip).round(3)
df_catcher.head()

,year,position,name,team,detail_position,game,gama_starter,inning,error,po,assist,double_play,fpct,pb,sb,cs,cs_pct
0,2022,포수,이지영,키움,포수,137,102,994.667,11,818,77,9,0.988,4,67,33,33.0
1,2022,포수,유강남,LG,포수,134,115,1008.333,7,821,71,8,0.992,4,91,19,17.3
2,2022,포수,박세혁,두산,포수,126,110,884.000,5,689,68,10,0.993,7,74,21,22.1
3,2022,포수,박동원,KIA,포수,114,98,865.000,5,703,71,8,0.994,5,40,22,35.5
4,2022,포수,최재훈,한화,포수,106,101,853.667,3,644,78,10,0.996,7,73,34,31.8


In [79]:
df = pd.DataFrame(df_batter_offence2)
df_copy=df.copy()
df_offence2 = df_copy.drop(['mh','ph_ba'], axis=1)
df_offence2 = df_offence2.rename(columns = {'gdp' : 'double_play'})
df_offence2 = df_offence2.drop(columns=['Unnamed: 0'])
df_offence2 = df_offence2.replace('-', np.nan)
df_offence2[ df_offence2.isnull().any(axis=1) ]
df_offence2 = df_offence2.dropna()
df_offence2.head()

,year,position,name,team,avg,bb,ibb,hbp,so,double_play,slg,obp,ops,risp
0,2022,포수,이병헌,삼성,0.750,0,0,0,1,0,1.000,0.750,1.750,0.500
1,2022,포수,권다결,KIA,0.500,0,0,0,0,0,0.500,0.500,1.000,0.500
2,2022,포수,김재성,삼성,0.335,18,0,2,40,4,0.453,0.402,0.855,0.311
3,2022,포수,안승한,두산,0.333,2,0,0,12,0,0.417,0.368,0.785,0.385
4,2022,포수,김선우,KIA,0.333,0,0,1,0,0,0.333,0.500,0.833,0.000


In [ ]:
df = pd.DataFrame(df_pitcher)
df_pitcher = df.copy()
df_pitcher[ df_pitcher.isin(['-']).any(axis=1) ]
df_pitcher = df_pitcher.drop(columns=['Unnamed: 0', 'wpct'])
df_pitcher['inning'] = df_pitcher['inning'].apply(convert_ip).round(3)
df_pitcher['position'] = '투수'
df_pitcher['detail_position'] = '투수'
df_pitcher.head()

,year,name,team,era,game_cnt,win,lose,save,hold,inning,hit,home_run,bb,hbp,kk,r,er,whip,position,detail_position
0,2022,안우진,키움,2.11,30,15,8,0,0,196.000,131,4,55,4,224,51,46,0.95,투수,투수
1,2022,김광현,SSG,2.13,28,13,3,0,0,173.333,141,10,45,5,153,48,41,1.07,투수,투수
2,2022,플럿코,LG,2.39,28,15,5,0,0,162.000,125,13,38,2,149,53,43,1.01,투수,투수
3,2022,수아레즈,삼성,2.49,30,6,8,0,0,173.667,151,7,50,4,159,61,48,1.16,투수,투수
4,2022,켈리,LG,2.54,27,16,4,0,0,166.333,144,10,35,2,153,50,47,1.08,투수,투수


In [94]:
# df_team
# df_offence
# df_offence2
# df_fielder
# df_catcher
# df_pitcher
fielder_players = df_fielder[['name', 'position', 'detail_position']]
catcher_players = df_catcher[['name', 'position', 'detail_position']]
pitcher_players = df_pitcher[['name', 'position', 'detail_position']]
players = pd.concat([fielder_players, catcher_players, pitcher_players])
players


,name,position,detail_position
46,양석환,내야수,1루수
47,김민성,내야수,3루수
48,김주원,내야수,유격수
49,정훈,내야수,1루수
50,오영수,내야수,1루수
...,...,...,...
92,사우어,투수,투수
93,화이트,투수,투수
94,에르난데스,투수,투수
95,로드리게스,투수,투수


In [103]:
teams = pd.DataFrame(df_pitcher['team'].unique())
teams.columns = ['team_name']
teams

,team_name
0,키움
1,SSG
2,LG
3,삼성
4,NC
5,KT
6,두산
7,롯데
8,KIA
9,한화


In [33]:
load_dotenv()

DB_CONFIG = {
    'host' : os.getenv('HOST', 'localhost'),
    'user' : os.getenv('USER', 'root'),
    'password' : os.getenv('PASSWORD'),
    'port' : os.getenv('PORT', 3306),
    'database' : 'kbo_1'
}

db = DBManager(**DB_CONFIG)
db.DBconnect()

query = "CREATE TABLE IF NOT EXISTS `kbo_1`.`players` (\
  `player_id` INT NOT NULL AUTO_INCREMENT,\
  `name` VARCHAR(45) NOT NULL,\
  `position` VARCHAR(45) NOT NULL,\
  `detail_position` VARCHAR(45) NULL,\
  PRIMARY KEY (`player_id`),\
  UNIQUE INDEX `player_id_UNIQUE` (`player_id` ASC) VISIBLE)\
ENGINE = InnoDB;"
db.DBexecute(query)

query = "CREATE TABLE IF NOT EXISTS `kbo_1`.`teams` (\
  `team_id` INT NOT NULL AUTO_INCREMENT,\
  `team_name` VARCHAR(45) NOT NULL,\
  PRIMARY KEY (`team_id`),\
  UNIQUE INDEX `team_name_UNIQUE` (`team_name` ASC) VISIBLE)\
ENGINE = InnoDB;"
db.DBexecute(query)

db.DBclose()


DB연결 완료
연결 종료


In [96]:
def insert_data(df, table_name):

    columns = list(df.columns)
    columns_str = ', '.join([f'`{col}`' for col in columns])
    
    values_str = ', '.join(['%s'] * len(columns))

    return f"INSERT INTO {table_name} ({columns_str}) \nVALUES ({values_str})"

In [104]:
db = DBManager(**DB_CONFIG)
db.DBconnect()

query = insert_data(players, 'players')
for col, row in players.iterrows():
    db.DBexecute(query, tuple(row))

query = insert_data(teams, 'teams')
for col, row in teams.iterrows():
    db.DBexecute(query, tuple(row))

db.DBclose()

DB연결 완료
연결 종료


In [121]:
# team_stats table
db = DBManager(**DB_CONFIG)
db.DBconnect()
result = db.DBSelect("SELECT team_id, team_name FROM teams")
db.DBclose()
team_db = pd.DataFrame(result, columns=['team_id', 'team_name'])

df_merged_team = pd.merge(
    df_team,
    team_db,
    left_on='team',
    right_on='team_name',
    how='left'
)
df_merged_team
teams = df_merged_team.drop(columns=['team'])
teams = teams.drop(columns=['team_name'])

teams.head()

DB연결 완료
연결 종료


,year,ranking,game,win,lose,draw,winning_rate,games_behind,recent10,inarow,home,away,team_id
0,2022,1,144,88,52,4,0.629,0.0,4승0무6패,4패,49-0-23,39-4-29,2
1,2022,2,144,80,62,2,0.563,9.0,5승0무5패,1승,39-1-32,41-1-30,1
2,2022,3,144,87,55,2,0.613,2.0,4승0무6패,1승,38-1-33,49-1-22,3
3,2022,4,144,80,62,2,0.563,9.0,7승0무3패,1패,40-1-31,40-1-31,6
4,2022,5,144,70,73,1,0.490,19.5,7승0무3패,1패,31-0-41,39-1-32,9
